In [1]:
from rdflib  import Graph
import owlrl
from pathlib import Path

PROJECT_ROOT  = Path.home() / "xai-knowledge-graph"
ONTOLOGY_PATH = PROJECT_ROOT / "ontology" / "xai-kg.ttl"
DATA_PATH     = PROJECT_ROOT / "data" / "rdf" / "xai_kg_data.ttl"

g = Graph()
g.parse(ONTOLOGY_PATH, format="turtle")
print(f"Loaded ontology: {len(g):,} triples")

g.parse(DATA_PATH, format="turtle")
print(f"Loaded data:     {len(g):,} triples total")

Loaded ontology: 80 triples
Loaded data:     87,358 triples total


In [2]:
print(f"Triples BEFORE reasoning: {len(g):,}")

owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)

print(f"Triples AFTER  reasoning: {len(g):,}")
print(f"New triples derived:      {(len(g) - 87358):,}  (approximate)")  # adjust based on your Cell 1 count

Triples BEFORE reasoning: 87,358
Triples AFTER  reasoning: 261,755
New triples derived:      174,397  (approximate)


In [3]:
g.serialize(destination=str(PROJECT_ROOT / "data" / "rdf" / "xai_kg_reasoned.ttl"), format="turtle")
print("Reasoned graph saved.")

Reasoned graph saved.


## SPARQL queries

In [4]:
query = """
PREFIX : <http://xai-kg.example.org/>
PREFIX foaf:    <http://xmlns.com/foaf/0.1/>
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?title ?year ?citations
WHERE {
    ?paper a :Paper ;
           :title ?title ;
           :year ?year ;
           :citationCount ?citations .
}
ORDER BY DESC(?citations)
LIMIT 10
"""
for row in g.query(query):
    print(f"[{row.citations:>6}] ({row.year}) {row.title[:80]}")

[ 26975] (2016) Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization
[  5264] (2017) Explanation in Artificial Intelligence: Insights from the Social Sciences
[  3132] (2017) Grad-CAM++: Improved Visual Explanations for Deep Convolutional Networks
[  1817] (2018) Consistent Individualized Feature Attribution for Tree Ensembles
[  1089] (2019) Fooling LIME and SHAP: Adversarial Attacks on Post hoc Explanation Methods
[  1012] (2021) Explainable artificial intelligence (XAI) in deep learning-based medical image a
[   936] (2020) Questioning the AI: Informing Design Practices for Explainable AI User Experienc
[   908] (2018) Visual Interpretability for Deep Learning: a Survey
[   749] (2020) Opportunities and Challenges in Explainable Artificial Intelligence (XAI): A Sur
[   671] (2022) From Anecdotal Evidence to Quantitative Evaluation Methods: A Systematic Review 


In [5]:
query="""
PREFIX : <http://xai-kg.example.org/>
PREFIX foaf:    <http://xmlns.com/foaf/0.1/>
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?authorName (COUNT(?paper) AS ?paperCount)
WHERE{
    ?author a :Author ;
            :name ?authorName ;
            :authored ?paper.
}
GROUP BY ?authorName
ORDER BY DESC(?paperCount)

LIMIT 10
"""

print(f"{'Author':<30} {'Papers':>6}")
print("-" * 38)
for row in g.query(query):
    print(f"{row.authorName:<30}  {row.paperCount:>6}")

Author                         Papers
--------------------------------------
S. Lapuschkin                       23
W. Samek                            17
Barbara Hammer                      16
David Martens                       14
João Marques-Silva                  12
P. Biecek                           12
Xuanxiang Huang                     12
Eyke Hüllermeier                    11
Francesca Toni                      11
Inga Strümke                        11


In [6]:
query = """
PREFIX : <http://xai-kg.example.org/>

SELECT ?topicName (COUNT(?paper) AS ?paperCount)
WHERE {
    ?paper :about ?topic .
    ?topic :name ?topicName .
}
GROUP BY ?topicName
ORDER BY DESC(?paperCount)
LIMIT 20
"""

print(f"{'Topic':<25} {'Papers':>7}")
print("-" * 33)
for row in g.query(query):
    print(f"{str(row.topicName):<25} {row.paperCount:>7}")

Topic                      Papers
---------------------------------
Explainability               2240
Interpretability             1817
SHAP                         1057
Healthcare                    903
Transparency                  582
NLP                           483
Feature Attribution           476
Computer Vision               442
Counterfactual                405
Grad-CAM                      395
LIME                          364
Saliency                      330
Post-hoc                      298
Trust                         272
Model-Agnostic                232
Integrated Gradients          167
Attention                     153
Time Series                   151
Fairness                      121
Tabular                       116


In [7]:
query="""
PREFIX : <http://xai-kg.example.org/>
SELECT DISTINCT ?authorName
WHERE {
    ?author :worksOn ?topic ;
            :name  ?authorName .
    ?topic  :name "SHAP" .
}
ORDER BY ?authorName

"""
results = list(g.query(query))

print(f"Authors working on SHAP: {len(results)}\n")

for r in results:
    print(r.authorName)

Authors working on SHAP: 4009

A. Abdallah
A. Abdollahi
A. Abdullin
A. Al-Manea
A. Alhassan
A. Amassian
A. Amin
A. Anžel
A. Argha
A. Arikan
A. Azim
A. Bedayat
A. Bellachia
A. Beltran-Bless
A. Bihorac
A. Bogdanova
A. Bonfigli
A. Boukabou
A. Bunea
A. Burden
A. Bâra
A. C. Lorena
A. Cagol
A. Carlo
A. Chaddad
A. Chaurasia
A. Chehri
A. Cisnal
A. Cutler
A. Dehoux
A. Dengel
A. Depeursinge
A. Dumitrascu
A. Ehsan
A. F. Jouzdani
A. Fania
A. Farabi
A. Farimani
A. Fitzpatrick
A. Folarin
A. G. Bellotti
A. G. Luminare
A. Giuliani
A. Haberl
A. Hammer
A. Hasenburg
A. Huq
A. Imakura
A. Jain
A. Jansma
A. Jemaa
A. John
A. Jung
A. Jurcut
A. Jøsang
A. Kar
A. Karunananda
A. Khan
A. Kikuchi
A. Krishnan
A. Lacalamita
A. Lekkas
A. Lykov
A. Maglione
A. Mahasinghe
A. Malhi
A. Manjavacas
A. Masoomi
A. Mastropaolo
A. Meroni
A. Meroz
A. Meyer
A. Michala
A. Mishra
A. Monaco
A. Mondal
A. Monreale
A. Nadeem
A. Nguyen-Tuong
A. Nourbakhsh
A. Nugaliyadde
A. Owen
A. O’Sullivan
A. Pedrocchi
A. Perotti
A. Pouria
A. Prelaj
A.

In [8]:
from rdflib import Graph
from pathlib import Path

# Load fresh graph — no reasoning
PROJECT_ROOT = Path.home() / "xai-knowledge-graph"
g_no_reasoning = Graph()
g_no_reasoning.parse(PROJECT_ROOT / "ontology" / "xai-kg.ttl", format="turtle")
g_no_reasoning.parse(PROJECT_ROOT / "data" / "rdf" / "xai_kg_data.ttl", format="turtle")
print(f"Triples (no reasoning): {len(g_no_reasoning):,}")

# Run THE SAME query
results_no_reasoning = list(g_no_reasoning.query(query))
print(f"Authors working on SHAP (no reasoning): {len(results_no_reasoning)}")

# Compare
print(f"\nReasoned graph result:    {len(results)} authors")
print(f"Un-reasoned graph result: {len(results_no_reasoning)} authors")

Triples (no reasoning): 87,358
Authors working on SHAP (no reasoning): 0

Reasoned graph result:    4009 authors
Un-reasoned graph result: 0 authors


In [9]:
query = """
PREFIX : <http://xai-kg.example.org/>

SELECT ?title ?year ?citations
WHERE {
    ?paper :title ?title ;
           :year ?year ;
           :citationCount ?citations ;
           :about ?topic1 ;
           :about ?topic2 .
    ?topic1 :name "Interpretability" .
    ?topic2 :name "Computer Vision" .
}
ORDER BY DESC(?citations)
LIMIT 10
"""

print(f"{'Citations':>10}  {'Year':>4}  Title")
print("-" * 100)
for row in g.query(query):
    print(f"{int(row.citations):>10,}  {int(row.year):>4}  {row.title}")
    

 Citations  Year  Title
----------------------------------------------------------------------------------------------------
     3,132  2017  Grad-CAM++: Improved Visual Explanations for Deep Convolutional Networks
       908  2018  Visual Interpretability for Deep Learning: a Survey
       196  2020  Towards Interpretable Semantic Segmentation via Gradient-weighted Class Activation Mapping
       139  2022  Explainable Deep Learning Methods in Medical Image Classification: A Survey
       136  2021  Explainable Artificial Intelligence for Human Decision-Support System in Medical Domain
       115  2021  Improving Deep Learning Interpretability by Saliency Guided Training
        86  2022  Fault Diagnosis using eXplainable AI: a Transfer Learning-based Approach for Rotating Machinery exploiting Augmented Synthetic Data
        78  2023  Is Grad-CAM Explainable in Medical Images?
        77  2019  Deep neural network or dermatologist?
        62  2021  Evaluation of Interpretability fo